# 06 Model selection and hyperparameter tuning

We predict school AP participation in districts held out during training. Run notebook 04 first to create the development split. Notebook 05 compared missing-value strategies. Here we compare model families using the same five district folds. We select the configuration with the lowest mean validation RMSE. RMSE penalizes large errors more heavily. We also report mean absolute error (MAE) in percentage points.

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, ParameterGrid
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from xgboost import XGBRegressor

import preprocessing_utils as pp

data = joblib.load("../data/processed/04_preprocessing.joblib")
X_dev, y_dev, districts = data["X_other"], data["y_other"], data["groups_other"]
columns = [data["feature_types"][key] for key in ("cont_skewed", "cont_other", "cat_ftrs", "ord_ftrs")]
assert {"ap_rate", "enrl_AP", "enrl_9to12", "leaid"}.isdisjoint(X_dev.columns)
seed = 0

def make_prep(strategy):
    if strategy == "imputed":
        return pp.build_imputation_preprocessor(*columns, seed=seed, n_estimators=10, max_iter=12)
    return pp.build_preprocessor(*columns)

## Candidate models

The baseline predicts the mean AP rate of the schools in each training fold. Ridge, ElasticNet, and KNN use random forest iterative imputation. Random forest and XGBoost handle missing values directly. Set `include_svr` to True to include the slower RBF model.

"Model name": (
    estimator,
    preprocessing_strategy,
    hyperparameter_grid
)

In [2]:
include_svr = False
alphas = np.logspace(-3, 3, 13)
models = {
    "Training mean": (DummyRegressor(strategy="mean"), "native", {}),
    "Ridge": (Ridge(), "imputed", {"alpha": alphas}),
    "ElasticNet": (ElasticNet(max_iter=10000), "imputed",
                   {"alpha": alphas, "l1_ratio": [0.1, 0.5, 0.9]}),
    "Random forest": (RandomForestRegressor(n_estimators=100, random_state=seed, n_jobs=2),
                      "native", {"max_depth": [5, 10, 20, None], "max_features": [0.3, 0.5, 1.0],
                                 "min_samples_leaf": [1, 5, 20]}),
    "XGBoost": (XGBRegressor(n_estimators=500, eval_metric="rmse", random_state=seed, n_jobs=2),
                "native", {"max_depth": [3, 5, 7], "learning_rate": [0.03, 0.1],
                           "subsample": [0.7, 1.0], "colsample_bytree": [0.7, 1.0],
                           "reg_lambda": [1, 10]}),
    "KNN": (KNeighborsRegressor(), "imputed",
            {"n_neighbors": [5, 15, 30, 50], "weights": ["uniform", "distance"]}),
}
if include_svr:
    models["SVR"] = (SVR(kernel="rbf"), "imputed",
                     {"C": [0.1, 1, 10], "gamma": ["scale", 0.01, 0.1]})

## Tune on development districts

Within each fold, we fit the two preprocessors once on the training schools and reuse their outputs for every candidate. For XGBoost, we first hold out 20% of the training districts to choose the number of trees. We fit a separate preprocessor for that step, then refit XGBoost on the full training fold before scoring it.

In [3]:
scores = []
for fold, (train, valid) in enumerate(GroupKFold(5).split(X_dev, groups=districts), start=1):
    X_train, y_train = X_dev.iloc[train], y_dev.iloc[train]
    X_valid, y_valid = X_dev.iloc[valid], y_dev.iloc[valid]
    assert set(districts.iloc[train]).isdisjoint(districts.iloc[valid])
    matrices = {}
    for strategy in ["native", "imputed"]:
        prep = make_prep(strategy)
        matrices[strategy] = (prep.fit_transform(X_train), prep.transform(X_valid))

    split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    grow, stop = next(split.split(X_train, groups=districts.iloc[train]))
    inner_prep = make_prep("native")
    X_grow = inner_prep.fit_transform(X_train.iloc[grow])
    X_stop = inner_prep.transform(X_train.iloc[stop])

    for name, (estimator, strategy, grid) in models.items():
        train_matrix, valid_matrix = matrices[strategy]
        for candidate, params in enumerate(ParameterGrid(grid)):
            model = clone(estimator).set_params(**params)
            trees = None
            if name == "XGBoost":
                trial = clone(model).set_params(early_stopping_rounds=20)
                trial.fit(X_grow, y_train.iloc[grow], eval_set=[(X_stop, y_train.iloc[stop])],
                          verbose=False)
                trees = trial.best_iteration + 1
                model.set_params(n_estimators=trees)
            model.fit(train_matrix, y_train)
            prediction = model.predict(valid_matrix)
            scores.append({"model": name, "candidate": candidate, "fold": fold, "trees": trees,
                           "rmse": root_mean_squared_error(y_valid, prediction),
                           "mae_pp": 100 * mean_absolute_error(y_valid, prediction)})
    print(f"Completed fold {fold} of 5")

Completed fold 1 of 5
Completed fold 2 of 5
Completed fold 3 of 5
Completed fold 4 of 5
Completed fold 5 of 5


Grow set: trains the XGBoost trial model.
Stop set: determines the optimal number of trees.
Validation set: evaluates the hyperparameter combination.

## Select and refit

We show the best configuration for each family, including the training-mean baseline. The RMSE standard deviation measures variation across district folds. If XGBoost wins, we use the median number of trees selected in its five early-stopping runs for the development refit. The fitted pipeline is ready for a later final test evaluation.

In [4]:
fold_scores = pd.DataFrame(scores)
ranking = (fold_scores.groupby(["model", "candidate"], as_index=False)
           .agg(mean_rmse=("rmse", "mean"), sd_rmse=("rmse", "std"), mae_pp=("mae_pp", "mean"))
           .sort_values("mean_rmse"))
leaders = ranking.drop_duplicates("model").copy()
leaders["parameters"] = [ParameterGrid(models[row.model][2])[row.candidate]
                         for row in leaders.itertuples()]
print(leaders.drop(columns="candidate").to_string(index=False))

        model  mean_rmse  sd_rmse    mae_pp                                                                                           parameters
      XGBoost   0.121557 0.004769  7.857583 {'subsample': 0.7, 'reg_lambda': 10, 'max_depth': 7, 'learning_rate': 0.03, 'colsample_bytree': 0.7}
Random forest   0.122994 0.005645  7.986967                                      {'min_samples_leaf': 5, 'max_features': 0.3, 'max_depth': None}
          KNN   0.124843 0.004123  8.157688                                                           {'weights': 'distance', 'n_neighbors': 50}
        Ridge   0.129627 0.007445  8.743233                                                                        {'alpha': 31.622776601683793}
   ElasticNet   0.129795 0.007507  8.756099                                                                    {'l1_ratio': 0.1, 'alpha': 0.001}
Training mean   0.156850 0.005624 11.907943                                                                                       

In [5]:
winner = leaders.iloc[0]
estimator, strategy, grid = models[winner["model"]]
final_estimator = clone(estimator).set_params(**winner["parameters"])
if winner["model"] == "XGBoost":
    selected = fold_scores.query("model == 'XGBoost' and candidate == @winner.candidate")
    final_estimator.set_params(n_estimators=int(selected["trees"].median()))
selected_pipeline = make_pipeline(make_prep(strategy), final_estimator)
selected_pipeline.fit(X_dev, y_dev)
print(f"Selected by mean CV RMSE: {winner['model']}")

Selected by mean CV RMSE: XGBoost


### Model training and hyperparameter tuning

We used five-fold GroupKFold cross-validation to tune the hyperparameters of each model. We grouped the observations by school district to make sure that schools from the same district did not appear in both the training and validation sets.

For each fold, we fitted the preprocessing pipelines on the training data and applied the same transformations to the validation data. Models that support missing values used native missing-value handling, while the remaining models used multivariate imputation.

We used ParameterGrid to test every combination of hyperparameters and evaluated each combination using RMSE and MAE. For XGBoost, we created an additional group-aware split within the training fold to determine the optimal number of trees through early stopping. We then retrained the model on the entire training fold using that number of trees.

We selected the best hyperparameters for each model based on the lowest mean validation RMSE across the five folds and used the standard deviation to assess the consistency of its performance. The test set was kept separate throughout this process.